# Tree Traversals - DFS (Preorder / Inorder / Postorder) & BFS
**Topic:** Tree · **Type:** Concept notebook (not a single LeetCode problem)

Nearly every tree question is *some* traversal with extra bookkeeping bolted on. This notebook builds all
four orders from first principles, each one **recursively** and **iteratively**.

## 💡 Concepts

**Core concept(s):** **Traversal** - visiting every node of a tree exactly once, in a *defined order*.

**Why it matters:** A tree has no natural "first, second, third" the way a list does. Branching means several
orders are sensible, and *which order you pick decides which problems become easy*. Validate-BST wants inorder;
freeing a tree wants postorder; copying a tree wants preorder; "shortest number of hops" wants BFS.

**Key intuition:** All four traversals do the *same* work - touch every node once, `O(n)`. They differ only in
**when a node is recorded relative to its children**, and in **which container holds the work not yet done**:
a **stack** (DFS) or a **queue** (BFS).

---

### 📚 What is a Binary Tree?
A **binary tree** is a set of **nodes** in a branching shape. Each node holds a value and links to at most two
children, called **left** and **right**. The one node with no parent is the **root**; nodes with no children are
**leaves**. There are no cycles - from the root there is exactly one path down to any node.
- **In Python:** a tiny class with `.val`, `.left`, `.right`. `None` means "no child here".
- **Subtree:** any node *plus everything hanging below it* is itself a perfectly good tree. This is the single
  most useful fact about trees - it is why recursion works so cleanly here.

### 📚 What is a Traversal?
A **traversal** is an ordering of the nodes produced by a systematic walk. "Systematic" means no node is
skipped and no node is visited twice. The four standard orders are **preorder**, **inorder**, **postorder**
(all three are DFS) and **level order** (BFS).

### 📚 What is DFS (Depth-First Search)?
**DFS** goes as *deep* as it can down one branch before backtracking to try the next. Think of a maze walked
with your hand on the left wall: you commit to a corridor fully, hit a dead end, back up one junction, take the
next corridor.
- It needs a **LIFO** container (last in, first out) - the most recently discovered node is explored next.
- **Recursion gives you that container for free:** the language's own **call stack** *is* the stack.
- **Complexity:** `O(n)` time (each node once); `O(h)` space, where `h` is the **height** - the stack only ever
  holds one root-to-current path. Balanced tree -> `h ≈ log n`. A degenerate "linked list" tree -> `h = n`.

### 📚 What is a Stack?
A **stack** is a pile: you `push` on top and `pop` from the top, so the **last** thing in is the **first** out
(**LIFO**). In Python a plain `list` is a stack: `stack.append(x)` and `stack.pop()`, both `O(1)`.
- **The call stack:** each active function call is a frame stacked on the previous one. When a function calls
  itself, a new frame goes on top; when it returns, the frame pops and execution resumes *exactly where it left
  off*. That "resume where it left off" is what makes recursive traversal look effortless - and it is exactly
  what you must recreate by hand in the iterative versions.

### 📚 What is BFS (Breadth-First Search)?
**BFS** explores in rings around the root: everything at distance 0, then distance 1, then distance 2... It
finishes an entire **level** before touching the next one.
- It needs a **FIFO** container - a **queue**.
- **Complexity:** `O(n)` time; `O(w)` space, where `w` is the tree's **maximum width**. In a full binary tree the
  bottom level alone holds about `n/2` nodes, so BFS space is `O(n)`.

### 📚 What is a Queue?
A **queue** is a checkout line: you enqueue at the back and dequeue from the front - **first in, first out**
(**FIFO**). Use `collections.deque`, where `append()` (back) and `popleft()` (front) are both `O(1)`.
> ⚠️ Never use a plain `list` with `list.pop(0)` as a queue - that shifts every remaining element, making it
> `O(n)` per pop and `O(n²)` overall.

---

### The three DFS orders, in one line each
At each node you do three things: **visit the node (N)**, **recurse left (L)**, **recurse right (R)**. The order
you write those three lines *is* the traversal:

| Order | Sequence | Records a node... | Classic use |
|---|---|---|---|
| **Preorder** | `N L R` | **before** its children | copy / serialize a tree; print a folder name before its contents |
| **Inorder** | `L N R` | **between** its two subtrees | **BST -> sorted output**, validate BST, k-th smallest |
| **Postorder** | `L R N` | **after** its children | delete/free a tree; any value computed *from* children (height, max path sum) |
| **Level order (BFS)** | ring by ring | grouped by depth | level averages, right-side view, shortest hops, zigzag |

**Mnemonic:** *pre / in / post* says where the **root** sits relative to its subtrees. Left always precedes
right in all three.

## 📝 The Running Example

Every traversal below walks this **perfect** tree, so the outputs are directly comparable:

```
            1
          /   \
         2     3
        / \   / \
       4   5 6   7
```

| Traversal | Output | Read it as |
|---|---|---|
| Preorder `N L R` | `1 2 4 5 3 6 7` | root, then the whole left subtree, then the whole right |
| Inorder `L N R` | `4 2 5 1 6 3 7` | left subtree, root, right subtree - the root sits dead centre |
| Postorder `L R N` | `4 5 2 6 7 3 1` | children always before their parent; the **root is last** |
| BFS level order | `1 2 3 4 5 6 7` | row by row, left to right |

Three sanity checks worth memorising: **preorder always starts at the root**, **postorder always ends at the
root**, and **inorder on a BST is sorted**.

In [1]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

    def __repr__(self):                    # makes printed stacks/queues readable in traces
        return f"({self.val})"


def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root


def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) - used by the benchmark at the end."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left  = helper(lo, mid - 1)   # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)


def show(root, prefix="", side="root"):
    """Print the tree sideways so its shape is obvious (right child drawn above)."""
    if not root:
        return
    show(root.right, prefix + "     ", "R")           # right subtree first -> appears on top
    print(f"{prefix}{root.val}  <- {side}")
    show(root.left, prefix + "     ", "L")


TREE = build_tree([1, 2, 3, 4, 5, 6, 7])              # the running example
BST  = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])   # used for the inorder demo

print("TREE, drawn sideways (tilt your head left):")
show(TREE)

TREE, drawn sideways (tilt your head left):
          7  <- R
     3  <- R
          6  <- L
1  <- root
          5  <- R
     2  <- L
          4  <- L


---
# Part 1 - DFS

All three DFS orders come out of the *same* three-line body. Only the line order changes:

```python
def dfs(node):
    if not node: return      # base case: fell off the bottom of the tree
    visit(node)              # N   <-- move this one line to change the traversal
    dfs(node.left)           # L
    dfs(node.right)          # R
```

The recursive versions are almost too short to be interesting. The **iterative** versions are where the
understanding lives - and where interviewers push, because *"now do it without recursion"* is the standard
follow-up.

## 1.1 Preorder - `N L R`

**Idea:** record a node the *instant* you arrive, then go left, then go right.

**Why you'd want it:** the output doubles as a build-instruction list. The first element is the root, so you can
rebuild or clone the tree by reading left to right. That is why serialization uses preorder.

**Time:** `O(n)` - each node visited once.
**Space:** `O(h)` - the call stack holds one root-to-node path.

In [2]:
def preorder_recursive(root: Optional[TreeNode]) -> List[int]:
    """N L R - append the node BEFORE descending into either child."""
    out = []

    def dfs(node):
        if not node:
            return                         # base case: an empty subtree contributes nothing
        out.append(node.val)               # N - record on ARRIVAL
        dfs(node.left)                     # L - finish the entire left subtree first
        dfs(node.right)                    # R - then the entire right subtree

    dfs(root)
    return out


print(preorder_recursive(TREE))            # expect [1, 2, 4, 5, 3, 6, 7]

[1, 2, 4, 5, 3, 6, 7]


### Preorder, iterative

Preorder is the **only** order that maps onto a stack with no bookkeeping, because the moment you pop a node you
are already allowed to record it - nothing about its children has to happen first.

**The one trick:** a stack reverses what you push. You want **left** handled before **right**, so push
**right first, left second** - left lands on top and pops next.

```
pop 1  -> out=[1]      push right(3), then left(2)   stack=[3, 2]
pop 2  -> out=[1,2]    push 5, then 4                stack=[3, 5, 4]
pop 4  -> out=[1,2,4]  leaf, nothing to push         stack=[3, 5]
```

**Time:** `O(n)`. **Space:** `O(h)` - the stack mirrors the recursion.

In [3]:
def preorder_iterative(root: Optional[TreeNode]) -> List[int]:
    
    """N L R with an explicit stack. Record on pop; push RIGHT before LEFT."""
    if not root:
        return []
    out, stack = [], [root]                # stack = subtrees discovered but not yet processed
    while stack:
        node = stack.pop()                 # LIFO: the most recently pushed node comes out first
        out.append(node.val)               # N - safe to record immediately
        if node.right:
            stack.append(node.right)       # pushed FIRST  -> sits DEEPER -> popped LATER
        if node.left:
            stack.append(node.left)        # pushed SECOND -> sits on TOP -> popped NEXT
    return out


print(preorder_iterative(TREE))            # expect [1, 2, 4, 5, 3, 6, 7]

[1, 2, 4, 5, 3, 6, 7]


## 1.2 Inorder - `L N R`

**Idea:** a node may only be recorded once its **entire left subtree** is done, and before any of its right
subtree.

**Why you'd want it:** on a **BST** (left subtree < node < right subtree), inorder emits values in **sorted
order**. That single fact powers validate-BST, k-th-smallest, BST-to-sorted-list, and "find the in-order
successor".

**Time:** `O(n)`. **Space:** `O(h)`.

In [4]:
def inorder_recursive(root: Optional[TreeNode]) -> List[int]:
    """L N R - append the node BETWEEN its two subtrees."""
    out = []

    def dfs(node):
        if not node:
            return
        dfs(node.left)                     # L - everything smaller / to the left, first
        out.append(node.val)               # N - record on the way back UP, mid-way
        dfs(node.right)                    # R - then everything to the right

    dfs(root)
    return out


print("TREE:", inorder_recursive(TREE))    # expect [4, 2, 5, 1, 6, 3, 7]
print("BST :", inorder_recursive(BST))     # sorted! [1, 3, 4, 6, 7, 8, 10, 13, 14]

TREE: [4, 2, 5, 1, 6, 3, 7]
BST : [1, 3, 4, 6, 7, 8, 10, 13, 14]


### Inorder, iterative

Now the stack has to do real work. You cannot record a node on arrival - its left subtree owes you output first.
So the loop has **two alternating modes**:

1. **Dive left.** From the current node, push it and keep walking left, pushing every node on the way, until you
   fall off (`node is None`). The stack now holds the whole left spine, deepest node on top.
2. **Pop and turn right.** Pop the top node - its left subtree is provably finished, because you only got past
   it by falling off the bottom. Record it (`N`), then set `node = popped.right` and go back to step 1 for that
   right subtree.

The loop keeps going while **either** there is a node to dive from **or** the stack still holds unfinished
ancestors - hence `while node or stack`.

**This "dive left, pop, turn right" shape is the backbone of the iterative BST problems** (k-th smallest is
literally this loop with a counter and an early `return`).

**Time:** `O(n)` - each node is pushed once and popped once. **Space:** `O(h)`.

In [5]:
def inorder_iterative(root: Optional[TreeNode]) -> List[int]:
    """L N R with an explicit stack: dive left, pop, record, turn right."""
    out, stack = [], []
    node = root                            # the subtree we are currently diving into
    while node or stack:
        while node:                        # 1) DIVE LEFT, remembering the path as we go
            stack.append(node)
            node = node.left               #    ...until we fall off the bottom-left
        node = stack.pop()                 # 2) POP: this node's LEFT subtree is now complete
        out.append(node.val)               # N - safe to record now
        node = node.right                  #    turn RIGHT; next loop dives into that subtree
                                           #    (if it's None, we just pop the next ancestor)
    return out


print("TREE:", inorder_iterative(TREE))    # expect [4, 2, 5, 1, 6, 3, 7]
print("BST :", inorder_iterative(BST))     # expect sorted

TREE: [4, 2, 5, 1, 6, 3, 7]
BST : [1, 3, 4, 6, 7, 8, 10, 13, 14]


## 1.3 Postorder - `L R N`

**Idea:** a node is recorded **last**, only after *both* of its subtrees are finished.

**Why you'd want it:** whenever a node's answer is **computed from its children's answers** - height, subtree
sum, "is this subtree balanced", binary-tree-maximum-path-sum, diameter. Also the only safe order in which to
free/delete nodes: you must not destroy a parent while its children are still reachable only through it.

**Time:** `O(n)`. **Space:** `O(h)`.

In [6]:
def postorder_recursive(root: Optional[TreeNode]) -> List[int]:
    """L R N - append the node AFTER both children are completely done."""
    out = []

    def dfs(node):
        if not node:
            return
        dfs(node.left)                     # L
        dfs(node.right)                    # R
        out.append(node.val)               # N - record last, on the way back up

    dfs(root)
    return out


print(postorder_recursive(TREE))           # expect [4, 5, 2, 6, 7, 3, 1]

[4, 5, 2, 6, 7, 3, 1]


### Postorder, iterative - method A: the reversal trick (easiest to remember)

Look at the two sequences:

```
postorder     L R N   ->  4 5 2 6 7 3 1
reversed      N R L   ->  1 3 7 6 2 5 4
```

Reversed postorder is exactly `N R L` - which is **preorder with left and right swapped**. So:

1. Run the plain preorder stack loop, but push **left first, right second** (mirrored), giving `N R L`.
2. **Reverse** the result.

Two tiny edits to code you already know. `list.reverse()` is `O(n)`, so the complexity is unchanged.
Equivalently, `append` to a `deque` with `appendleft` and skip the reversal.

**Time:** `O(n)`. **Space:** `O(h)` for the stack (plus the `O(n)` output every version needs).

In [7]:
def postorder_iterative_reverse(root: Optional[TreeNode]) -> List[int]:
    """L R N via the reversal trick: produce N R L with a stack, then reverse it."""
    if not root:
        return []
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.val)               # N - building N R L (mirrored preorder)
        if node.left:
            stack.append(node.left)        # push LEFT first  -> popped LATER
        if node.right:
            stack.append(node.right)       # push RIGHT second -> popped NEXT  => N R L
    out.reverse()                          # reverse(N R L) == L R N == postorder
    return out


print(postorder_iterative_reverse(TREE))   # expect [4, 5, 2, 6, 7, 3, 1]

[4, 5, 2, 6, 7, 3, 1]


### Postorder, iterative - method B: one stack, single pass (the "real" one)

The reversal trick is a lovely shortcut, but it does **not** visit nodes in postorder *as it runs* - it only
produces the list in postorder at the end. If you need to *act* on each node in true postorder (freeing memory,
streaming output), you need the genuine single-pass version.

The difficulty: when you peek at the top of the stack, you have to know **whether you already came back from its
right child**. Otherwise you'd descend right forever. The fix is one extra variable, `last_visited`:

- **Dive left** exactly like inorder, pushing the whole left spine.
- **Peek** at the top node:
  - if it has a right child **and that right child is not the node you just finished**, descend right;
  - otherwise both subtrees are done -> **pop it, record it**, and set `last_visited` to it.

`last_visited` is the hand-rolled equivalent of "which recursive call did I just return from".

**Time:** `O(n)`. **Space:** `O(h)`.

In [8]:
def postorder_iterative_one_stack(root: Optional[TreeNode]) -> List[int]:
    """L R N in a genuine single pass, using `last_visited` to avoid re-descending right."""
    out, stack = [], []
    node, last_visited = root, None        # last_visited = the node we most recently recorded
    while node or stack:
        while node:                        # 1) dive left, remembering the path
            stack.append(node)
            node = node.left
        peek = stack[-1]                   # 2) look at the deepest unfinished node
        # Descend right only if there IS a right child and we haven't just come back from it.
        if peek.right and last_visited is not peek.right:
            node = peek.right              #    right subtree still owes us output
        else:
            out.append(peek.val)           # N - both subtrees are done; record it
            last_visited = stack.pop()     #    remember it, so the parent knows we're back
    return out


print(postorder_iterative_one_stack(TREE))  # expect [4, 5, 2, 6, 7, 3, 1]

[4, 5, 2, 6, 7, 3, 1]


## 1.4 One template for all three - the "visited flag" stack

Three different iterative shapes is a lot to memorise under pressure. There is a **single template** that
produces any of the three by reordering three `push` lines - the same edit you make in the recursive version.

**The idea:** the reason iterative DFS is fiddly is that a node needs to be touched *twice* - once to expand its
children, once to record it. So store that state on the stack: push `(node, False)` meaning "not expanded yet"
and `(node, True)` meaning "children are already handled, just record me".

- Pop `(node, True)` -> record it, done.
- Pop `(node, False)` -> push its three pieces **in reverse of the order you want**, because a stack reverses.

For **inorder** (`L N R`) you push `right`, then `node(True)`, then `left` - so `left` pops first, then the node,
then `right`. ✅

**Time:** `O(n)`. **Space:** `O(h)` (`O(n)` in the worst case, since a node can sit on the stack twice).
Slightly slower than the specialised versions by a constant factor - the trade is one shape you never forget.

In [9]:
def dfs_universal(root: Optional[TreeNode], order: str = "inorder") -> List[int]:
    """Any DFS order from one loop. `visited=True` means 'children done - just record me'."""
    out, stack = [], [(root, False)]
    while stack:
        node, visited = stack.pop()
        if node is None:
            continue                       # skip empty children; keeps the pushes uncluttered
        if visited:
            out.append(node.val)           # N - this node's turn has finally come
            continue
        # Not expanded yet: push the three pieces in REVERSE of the desired order.
        if order == "preorder":            # want N L R  -> push R, L, N
            stack.append((node.right, False))
            stack.append((node.left,  False))
            stack.append((node,       True))
        elif order == "inorder":           # want L N R  -> push R, N, L
            stack.append((node.right, False))
            stack.append((node,       True))
            stack.append((node.left,  False))
        elif order == "postorder":         # want L R N  -> push N, R, L
            stack.append((node,       True))
            stack.append((node.right, False))
            stack.append((node.left,  False))
        else:
            raise ValueError(f"unknown order: {order!r}")
    return out


for o in ("preorder", "inorder", "postorder"):
    print(f"{o:>10}: {dfs_universal(TREE, o)}")

  preorder: [1, 2, 4, 5, 3, 6, 7]
   inorder: [4, 2, 5, 1, 6, 3, 7]
 postorder: [4, 5, 2, 6, 7, 3, 1]


## 1.5 Bonus - Morris inorder in `O(1)` space

Both recursion and an explicit stack cost `O(h)` memory. **Morris traversal** gets inorder in `O(n)` time and
**`O(1)` extra space** by temporarily rewiring the tree itself.

**The idea:** the node visited immediately before `node` in inorder is its **inorder predecessor** - the
right-most node of its left subtree. That predecessor's `right` pointer is `None` (that is what makes it
right-most), so borrow it: point it at `node`. Now, when the walk falls off the bottom of the left subtree, that
temporary "thread" carries you straight back up to `node` - no stack needed. On the second visit you find the
thread already in place, unlink it (restoring the tree exactly), and record the node.

Each edge is traversed at most three times, so it is still `O(n)`.

> Interview note: know that this exists and what it trades (it **mutates the tree mid-flight**, so it is unsafe
> with concurrent readers). Reach for it only when asked for `O(1)` space explicitly.

In [10]:
def inorder_morris(root: Optional[TreeNode]) -> List[int]:
    """L N R in O(1) extra space by threading each node to its inorder predecessor."""
    out, node = [], root
    while node:
        if not node.left:
            out.append(node.val)           # no left subtree -> nothing owed, record and go right
            node = node.right
        else:
            pred = node.left               # find the inorder PREDECESSOR: rightmost of left subtree
            while pred.right and pred.right is not node:
                pred = pred.right
            if pred.right is None:         # 1st visit: build the thread back up to `node`
                pred.right = node
                node = node.left           #            then explore the left subtree
            else:                          # 2nd visit: the thread is here, so left subtree is done
                pred.right = None          #            unlink it - the tree is restored
                out.append(node.val)       # N
                node = node.right
    return out


print(inorder_morris(TREE))                             # expect [4, 2, 5, 1, 6, 3, 7]
print(inorder_morris(BST))                              # sorted
print("tree restored:", inorder_recursive(TREE) == [4, 2, 5, 1, 6, 3, 7])

[4, 2, 5, 1, 6, 3, 7]
[1, 3, 4, 6, 7, 8, 10, 13, 14]
tree restored: True


---
# Part 2 - BFS (Level-Order Traversal)

Swap the stack for a **queue** and depth-first becomes breadth-first. That is genuinely the whole difference:

| | DFS | BFS |
|---|---|---|
| container | **stack** (LIFO) - `pop()` | **queue** (FIFO) - `popleft()` |
| explores | deepest discovered node next | *oldest* discovered node next |
| natural recursion? | yes | no - a queue has no call-stack equivalent |
| space | `O(h)` - one root-to-node path | `O(w)` - one full level, up to `~n/2` |
| good for | anything subtree-shaped | anything depth-shaped or shortest-hops |

**Why "oldest first" gives you levels:** a node's children are always enqueued *after* every node already
waiting, and children sit exactly one level deeper. So the queue is always sorted by depth, holding at most two
adjacent levels at a time.

## 2.1 Flat BFS

The bare loop: dequeue a node, record it, enqueue its children. Output is one flat list in level order.

**Time:** `O(n)`. **Space:** `O(w)` - the queue holds at most one level plus part of the next.

In [11]:
def bfs_flat(root: Optional[TreeNode]) -> List[int]:
    """Level order as one flat list."""
    if not root:
        return []
    out, q = [], deque([root])             # deque: append() at the back, popleft() at the front
    while q:
        node = q.popleft()                 # FIFO: the OLDEST discovered node - shallowest first
        out.append(node.val)
        if node.left:
            q.append(node.left)            # children land behind everything already waiting,
        if node.right:
            q.append(node.right)           # which is exactly why the queue stays depth-sorted
    return out


print(bfs_flat(TREE))                      # expect [1, 2, 3, 4, 5, 6, 7]

[1, 2, 3, 4, 5, 6, 7]


## 2.2 Level-order with levels grouped - the "level-size snapshot"

Most interview questions want the levels kept apart (`[[1], [2,3], [4,5,6,7]]`), not a flat list. The trick is
one line:

```python
for _ in range(len(q)):
```

**Take `len(q)` *before* the inner loop starts.** At that instant the queue holds *exactly* the current level and
nothing else. Freezing that count means the inner loop consumes precisely those nodes, while the children being
appended queue up for the next round. This snapshot is what turns BFS into "process one level at a time" - it is
the single most reused line in every BFS tree problem.

> Common bug: writing `while q:` for the inner loop, or letting the loop re-read `len(q)` as it grows. Either way
> the level boundary is lost.

**Time:** `O(n)`. **Space:** `O(w)`.

In [12]:
def bfs_levels(root: Optional[TreeNode]) -> List[List[int]]:
    """Level order, grouped one list per level - LeetCode 102."""
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        level_size = len(q)                # SNAPSHOT: q currently holds exactly this level
        level = []
        for _ in range(level_size):        # consume exactly that many - children queue up behind
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        out.append(level)                  # one finished level
    return out


print(bfs_levels(TREE))                    # expect [[1], [2, 3], [4, 5, 6, 7]]

[[1], [2, 3], [4, 5, 6, 7]]


## 2.3 Three variants that are just this loop plus one line

Once you own the level-size snapshot, a whole family of problems is a one-line edit:

- **Right-side view** ([LC 199](https://leetcode.com/problems/binary-tree-right-side-view/)): keep the **last** node of each level.
- **Zigzag / spiral** ([LC 103](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/)): reverse every other level.
- **Bottom-up level order** ([LC 107](https://leetcode.com/problems/binary-tree-level-order-traversal-ii/)): build normally, reverse the outer list at the end.

Others in the same family: level averages, largest value per level, minimum depth (return the moment you hit the
first leaf - BFS finds it earliest, which is exactly why BFS beats DFS for minimum depth).

All are `O(n)` time, `O(w)` space.

In [ ]:
def right_side_view(root: Optional[TreeNode]) -> List[int]:
    """What you'd see standing to the right: the LAST node of each level."""
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        n = len(q)
        for i in range(n):
            node = q.popleft()
            if i == n - 1:                 # the last node dequeued at this level
                out.append(node.val)
            if node.left:  
                q.append(node.left)
            if node.right: 
                q.append(node.right)
    return out


def zigzag(root: Optional[TreeNode]) -> List[List[int]]:
    """Level order, alternating left-to-right / right-to-left."""
    if not root:
        return []
    out, q, left_to_right = [], deque([root]), True
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:  
                q.append(node.left)
            if node.right: 
                q.append(node.right)
        out.append(level if left_to_right else level[::-1])   # flip every other level
        left_to_right = not left_to_right
    return out


def level_order_bottom_up(root: Optional[TreeNode]) -> List[List[int]]:
    """Deepest level first - build normally, then reverse."""
    levels = bfs_levels(root)
    levels.reverse()
    return levels


print("right side view :", right_side_view(TREE))
print("zigzag          :", zigzag(TREE))
print("bottom-up       :", level_order_bottom_up(TREE))

right side view : [1, 3, 7]
zigzag          : [[1], [3, 2], [4, 5, 6, 7]]
bottom-up       : [[4, 5, 6, 7], [2, 3], [1]]


## 2.4 The one thing BFS cannot fake

BFS has no natural recursive form, because recursion *is* a stack - it inherently goes deep. You can produce a
level-grouped result with DFS by passing the depth down and indexing into the output list, but the **visit order
is still depth-first**; only the *bucketing* is by level. That difference matters the moment you want to stop
early (minimum depth, first node matching X at the shallowest level) - only true BFS reaches shallow nodes first.

In [14]:
def levels_via_dfs(root: Optional[TreeNode]) -> List[List[int]]:
    """Level-GROUPED output from a DFS walk - same buckets, different visit order."""
    out = []

    def dfs(node, depth):
        if not node:
            return
        if depth == len(out):              # first time we reach this depth -> open a new bucket
            out.append([])
        out[depth].append(node.val)        # preorder arrival, filed under its depth
        dfs(node.left,  depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return out


print("BFS  :", bfs_levels(TREE))
print("DFS  :", levels_via_dfs(TREE))
print("same buckets:", bfs_levels(TREE) == levels_via_dfs(TREE))

# Same buckets, different VISIT order - compare the flat walks:
print()
print("BFS visit order:", bfs_flat(TREE))            # 1 2 3 4 5 6 7
print("DFS visit order:", preorder_recursive(TREE))    # 1 2 4 5 3 6 7
print("Node 3 sits at depth 1, yet DFS reaches it 4th - after two depth-2 nodes.")
print("That is why only true BFS can stop early at the SHALLOWEST match.")

BFS  : [[1], [2, 3], [4, 5, 6, 7]]
DFS  : [[1], [2, 3], [4, 5, 6, 7]]
same buckets: True

BFS visit order: [1, 2, 3, 4, 5, 6, 7]
DFS visit order: [1, 2, 4, 5, 3, 6, 7]
Node 3 sits at depth 1, yet DFS reaches it 4th - after two depth-2 nodes.
That is why only true BFS can stop early at the SHALLOWEST match.


## 2.5 When to reach for BFS vs. DFS - the pattern

Both visit every node in `O(n)`. The choice is never about correctness in general - it's about which one
**matches the shape of the question being asked**.

> **Scope note:** the worked examples below are kept to **tree** problems, to match this notebook's scope.
> The same decision rule applies unchanged to graphs and grids (multi-source BFS on a grid, backtracking
> over arrays, connected-component/cycle detection on a graph) - those get their own worked examples in a
> dedicated graph notebook later. See `1. bfs_dfs_pattern_playbook.ipynb` for the tree-only version of this
> same pattern, built out as reusable templates.

### The gut-check question

> **"Does the problem care about the *shortest* path, minimum steps, or fewest hops - in an *unweighted*
> tree/graph?"**
> - **Yes -> BFS.** BFS explores ring by ring, so the *first* time it reaches a target is guaranteed to be
>   via the shortest route. DFS can reach the same target through a long, winding path first and would have
>   to explore *everything* to be sure it found the shortest one.
> - **No (you need to explore full paths, compute something bottom-up from children, try every
>   combination, or just visit/mark every reachable node) -> DFS.** DFS is simpler to write recursively,
>   uses less memory on tall-narrow shapes (`O(h)` vs `O(w)`), and naturally accumulates "the path so far"
>   as it recurses - which path-enumeration needs and BFS cannot easily give you.

### Decision table

| Signal in the problem | Use | Why |
|---|---|---|
| "shortest path / minimum depth / fewest hops" (unweighted) | **BFS** | first arrival = shortest arrival |
| "level by level", "row by row", "nearest to the root" | **BFS** | the queue is always sorted by depth |
| a step can move in a direction `.left`/`.right` doesn't give you (e.g. toward the parent) | **BFS** (or DFS) works fine once you hand-build the missing edge (a parent map) - the algorithm doesn't change | trees only give you downward edges for free; BFS/DFS both just need *some* neighbour function |
| answer built **from children's answers** (height, sum, "is balanced", diameter) | **DFS (postorder)** | children must finish before the parent can compute anything |
| need the **full path**, not just reachability (root-to-leaf paths, path sums) | **DFS** | recursion carries "path so far" for free; BFS would need to store a path per queue entry (memory-heavy) |
| **path backtracking** - choose a child, recurse, un-choose, record at leaves | **DFS** | DFS *is* the call stack; undoing on the way back up is what backtracking means |
| answer needs **sorted order / BST validity / k-th smallest** | **DFS (inorder)** | inorder on a BST visits values in sorted order, for free |
| deep/skewed tree where recursion might blow the call stack | **BFS** (or iterative DFS) | BFS space is `O(w)`; if the tree is narrow this stays small even when it's tall |

*(Multi-source BFS on a grid, connected-component counting, and cycle detection are the same two
templates applied to graphs instead of trees - covered with their own worked examples in the graph
notebook.)*

### Worked examples (tree-shaped Blind 75 / NeetCode 150)

**BFS - because the question is secretly "shortest path in an unweighted tree":**
- **Binary Tree Level Order Traversal (102)** and **Minimum Depth of Binary Tree (111)** - minimum depth
  is *the* textbook reason BFS beats DFS: BFS stops the instant it meets the first leaf, guaranteed
  shallowest; DFS would have to check every path and keep a running minimum.
- **All Nodes Distance K in Binary Tree (863)** - "nodes exactly `k` hops away" also needs shallowest-first
  discovery, this time radiating outward (including *upward*, via a hand-built parent map) from a target
  node rather than downward from the root.

**DFS - because the question is paths, backtracking, or a bottom-up computation:**
- **Path Sum (112) / Binary Tree Paths (257) / Path Sum II (113) / Sum Root to Leaf Numbers (129) /
  Smallest String Starting From Leaf (988)** - need the *actual path* from root to leaf, which DFS builds
  up naturally as it recurses and un-builds on the way back.
- **Maximum Depth of Binary Tree (104), Diameter of Binary Tree (543), Balanced Binary Tree (110)** - each
  answer is computed *from the children's* answers -> postorder-style DFS.
- **Validate Binary Search Tree (98), Kth Smallest Element in a BST (230)** - need sorted order -> inorder
  DFS.

**One-line takeaway:** *unweighted shortest-path/minimum-hops -> BFS; everything else (paths, backtracking,
child-to-parent computation, sorted-order questions) -> DFS.* The full worked-out templates for every
problem above live in `1. bfs_dfs_pattern_playbook.ipynb`.


### Worked example: why BFS wins on "nearest leaf" (Minimum Depth of Binary Tree, LC 111)

In a tree there's only **one** path from the root to any given node, so DFS can't take a "longer road to the
same door" the way it can in a general graph. What it *can* do is check the **wrong door first** when several
candidate targets (here, several leaves) exist at different depths.

```
                 1
               /   \
              2       3      <- 3 is a LEAF at depth 1 (the true nearest leaf)
             / \
            4   5             <- 4 is a LEAF at depth 2
                 \
                  6            <- 6 is a LEAF at depth 3
```

- **DFS (preorder, pushing right before left so left pops first)** visits `1 -> 2 -> 4 -> 5 -> 6 -> 3`. If
  you stop at the *first* leaf you meet, that's `4` at **depth 3** - wrong. DFS fully commits to `2`'s
  subtree before it ever looks at `3`, so the actually-nearest leaf (`3`, depth 2) is checked *last*, not
  first.
- **BFS (level order)** visits `1`, then `2, 3` - and `3` is already a leaf. It stops immediately with the
  correct answer, **depth 2**. BFS can't possibly report anything deeper than the true nearest leaf, because
  it exhausts every shallower node before touching a deeper one.

This is precisely why LC 111 (Minimum Depth) is a BFS problem even though it's asked on a *tree*, not a
graph: "nearest" only means something when a container guarantees "shallowest discovered first," and only
BFS makes that guarantee mid-walk.


In [ ]:
TREE_MINDEPTH = build_tree([1, 2, 3, 4, 5, None, None, None, None, None, 6])
print("Tree for the minimum-depth example:")
show(TREE_MINDEPTH)


def min_depth_dfs_first_leaf(root: Optional[TreeNode]) -> int:
    \"\"\"NOT the real algorithm - deliberately buggy: returns the depth of whichever leaf
    preorder DFS happens to reach FIRST, to demonstrate why that is unsafe.\"\"\"
    if not root:
        return 0
    stack = [(root, 1)]
    while stack:
        node, depth = stack.pop()
        if not node.left and not node.right:      # first leaf popped - but NOT necessarily nearest
            return depth
        if node.right:
            stack.append((node.right, depth + 1))  # pushed first -> popped later (mirrors preorder)
        if node.left:
            stack.append((node.left, depth + 1))   # pushed second -> popped next
    return 0


def min_depth_bfs(root: Optional[TreeNode]) -> int:
    \"\"\"The correct, standard approach (LC 111): BFS returns the instant it meets the
    first leaf - guaranteed to be the shallowest one.\"\"\"
    if not root:
        return 0
    q = deque([(root, 1)])
    while q:
        node, depth = q.popleft()
        if not node.left and not node.right:       # first leaf DEQUEUED = nearest, guaranteed
            return depth
        if node.left:
            q.append((node.left, depth + 1))
        if node.right:
            q.append((node.right, depth + 1))
    return 0


print()
print("naive DFS 'first leaf found' depth:", min_depth_dfs_first_leaf(TREE_MINDEPTH), " <- WRONG (should be 2)")
print("BFS minimum depth                :", min_depth_bfs(TREE_MINDEPTH), " <- correct (2)")


---
# Part 3 - Watch the containers

Reading the code is one thing; watching the stack and queue evolve is what makes the difference stick. The two
traces below print the container **after every step**.

In [15]:
def trace_inorder(root):
    """Iterative inorder with the stack printed after every action."""
    out, stack, node, step = [], [], root, 0
    print(f"{'step':>4} | {'action':<22} | {'stack (bottom->top)':<24} | output")
    print(f"{'-'*4} | {'-'*22} | {'-'*24} | {'-'*20}")

    def log(action):
        print(f"{step:>4} | {action:<22} | {str(stack):<24} | {out}")

    while node or stack:
        while node:
            stack.append(node)
            step += 1; log(f"push {node.val}, go left")
            node = node.left
        node = stack.pop()
        out.append(node.val)
        step += 1; log(f"pop {node.val} -> record")
        node = node.right
    print("\ninorder =", out)


trace_inorder(TREE)

step | action                 | stack (bottom->top)      | output
---- | ---------------------- | ------------------------ | --------------------
   1 | push 1, go left        | [(1)]                    | []
   2 | push 2, go left        | [(1), (2)]               | []
   3 | push 4, go left        | [(1), (2), (4)]          | []
   4 | pop 4 -> record        | [(1), (2)]               | [4]
   5 | pop 2 -> record        | [(1)]                    | [4, 2]
   6 | push 5, go left        | [(1), (5)]               | [4, 2]
   7 | pop 5 -> record        | [(1)]                    | [4, 2, 5]
   8 | pop 1 -> record        | []                       | [4, 2, 5, 1]
   9 | push 3, go left        | [(3)]                    | [4, 2, 5, 1]
  10 | push 6, go left        | [(3), (6)]               | [4, 2, 5, 1]
  11 | pop 6 -> record        | [(3)]                    | [4, 2, 5, 1, 6]
  12 | pop 3 -> record        | []                       | [4, 2, 5, 1, 6, 3]
  13 | push 7, go left        | [(7

In [16]:
def trace_bfs(root):
    """Level-order BFS with the queue printed at every level boundary."""
    if not root:
        return
    q, level_no = deque([root]), 0
    print(f"{'level':>5} | {'queue at level start':<26} | {'size':>4} | nodes taken")
    print(f"{'-'*5} | {'-'*26} | {'-'*4} | {'-'*20}")
    while q:
        n = len(q)                                       # the snapshot
        print(f"{level_no:>5} | {str(list(q)):<26} | {n:>4} | ", end="")
        taken = []
        for _ in range(n):
            node = q.popleft()
            taken.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        print(taken)
        level_no += 1


trace_bfs(TREE)

level | queue at level start       | size | nodes taken
----- | -------------------------- | ---- | --------------------
    0 | [(1)]                      |    1 | [1]
    1 | [(2), (3)]                 |    2 | [2, 3]
    2 | [(4), (5), (6), (7)]       |    4 | [4, 5, 6, 7]


## ✅ Correctness check

Every iterative version is checked against its recursive twin, on shapes that break naive code: empty tree,
single node, left-skewed chain, right-skewed chain, a lopsided tree, and a random balanced BST.

In [17]:
import random

def right_chain(n):
    """1 -> 2 -> 3 ... all right children (worst case for the 'dive left' loops)."""
    root = TreeNode(1); cur = root
    for v in range(2, n + 1):
        cur.right = TreeNode(v); cur = cur.right
    return root

def left_chain(n):
    """n <- ... <- 2 <- 1, all left children (worst case for stack depth)."""
    root = TreeNode(1); cur = root
    for v in range(2, n + 1):
        cur.left = TreeNode(v); cur = cur.left
    return root

cases = {
    "empty":        None,
    "single":       build_tree([1]),
    "perfect(7)":   TREE,
    "bst(9)":       BST,
    "lopsided":     build_tree([1, 2, None, 3, None, 4]),
    "left chain":   left_chain(50),
    "right chain":  right_chain(50),
    "balanced(63)": build_balanced(63),
}

for name, root in cases.items():
    pre, ino, post = preorder_recursive(root), inorder_recursive(root), postorder_recursive(root)

    assert preorder_iterative(root)            == pre,  f"preorder iterative: {name}"
    assert dfs_universal(root, "preorder")     == pre,  f"preorder universal: {name}"

    assert inorder_iterative(root)             == ino,  f"inorder iterative: {name}"
    assert dfs_universal(root, "inorder")      == ino,  f"inorder universal: {name}"
    assert inorder_morris(root)                == ino,  f"inorder morris: {name}"

    assert postorder_iterative_reverse(root)   == post, f"postorder reverse: {name}"
    assert postorder_iterative_one_stack(root) == post, f"postorder one-stack: {name}"
    assert dfs_universal(root, "postorder")    == post, f"postorder universal: {name}"

    assert bfs_flat(root) == [v for lvl in bfs_levels(root) for v in lvl], f"bfs: {name}"
    assert bfs_levels(root) == levels_via_dfs(root), f"levels: {name}"

    print(f"{name:>13}: n={len(pre):>2}  pre={pre[:5]}{'...' if len(pre) > 5 else ''}")

# Extra invariants that must hold for ANY non-empty tree.
assert preorder_recursive(TREE)[0]   == TREE.val, "preorder must start at the root"
assert postorder_recursive(TREE)[-1] == TREE.val, "postorder must end at the root"
assert inorder_recursive(BST) == sorted(inorder_recursive(BST)), "inorder of a BST must be sorted"

# Random fuzzing against the recursive reference.
random.seed(0)
for _ in range(200):
    vals = [random.choice([random.randint(0, 9), None]) for _ in range(random.randint(1, 25))]
    vals[0] = random.randint(0, 9)                       # a valid tree needs a real root
    r = build_tree(vals)
    assert inorder_iterative(r)             == inorder_recursive(r)
    assert preorder_iterative(r)            == preorder_recursive(r)
    assert postorder_iterative_one_stack(r) == postorder_recursive(r)
    assert postorder_iterative_reverse(r)   == postorder_recursive(r)
    assert inorder_morris(r)                == inorder_recursive(r)

print("\nAll tests passed")

        empty: n= 0  pre=[]
       single: n= 1  pre=[1]
   perfect(7): n= 7  pre=[1, 2, 4, 5, 3]...
       bst(9): n= 9  pre=[8, 3, 1, 6, 4]...
     lopsided: n= 4  pre=[1, 2, 3, 4]
   left chain: n=50  pre=[1, 2, 3, 4, 5]...
  right chain: n=50  pre=[1, 2, 3, 4, 5]...
 balanced(63): n=63  pre=[32, 16, 8, 4, 2]...

All tests passed


## ⚠️ Why the iterative versions are not just showing off

Python's default recursion limit is ~1000 frames. A **skewed** tree - which is what you get from inserting sorted
data into an unbalanced BST, a very common real-world shape - has height `n`, so the recursive traversal blows up
while the iterative one is untroubled. The explicit stack lives on the heap, which is far larger than the call
stack.

In [18]:
import sys
print("recursion limit:", sys.getrecursionlimit())

deep = left_chain(5000)                    # height 5000 -> 5000 nested calls

try:
    inorder_recursive(deep)
    print("recursive: fine")
except RecursionError:
    print("recursive: RecursionError - blew the call stack")

vals = inorder_iterative(deep)             # same tree, explicit stack on the heap
print(f"iterative: fine - {len(vals)} nodes, first={vals[0]}, last={vals[-1]}")

recursion limit: 1000
recursive: RecursionError - blew the call stack
iterative: fine - 5000 nodes, first=5000, last=1


## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each traversal on trees of
growing size `n` and read the **doubling ratio** - how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` -> `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1x** |
| `O(n)`        | ≈ **2x** |
| `O(n²)`       | ≈ **4x** |

All six should land near **2x**: every traversal touches every node a constant number of times. What differs is
only the **constant factor** - recursion pays for function-call frames, the universal template pays for pushing
each node twice, Morris pays for re-walking threads. We use **balanced** trees (height ~log n) so the recursive
versions stay inside the recursion limit.

In [19]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)            # balanced -> every node visited, no early exit

solutions = {
    "preorder  recursive   O(n)": preorder_recursive,
    "preorder  iterative   O(n)": preorder_iterative,
    "inorder   recursive   O(n)": inorder_recursive,
    "inorder   iterative   O(n)": inorder_iterative,
    "inorder   morris      O(n)": inorder_morris,
    "postorder one-stack   O(n)": postorder_iterative_one_stack,
    "universal template    O(n)": lambda r: dfs_universal(r, "inorder"),
    "bfs       levels      O(n)": bfs_levels,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


preorder  recursive   O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.24 |           n/a
     2000 |       0.61 |         2.52x
     4000 |       1.85 |         3.03x
     8000 |       4.04 |         2.18x

preorder  iterative   O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.18 |           n/a
     2000 |       0.30 |         1.69x
     4000 |       1.20 |         3.95x
     8000 |       3.11 |         2.58x

inorder   recursive   O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.18 |           n/a
     2000 |       0.59 |         3.24x
     4000 |       1.13 |         1.93x
     8000 |       2.76 |         2.44x

inorder   iterative   O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.31 |           n/a
     2000 |       0.64 |         2.05x
     4000 |       0.98 |      

{'preorder  recursive   O(n)': [0.2425999998649786,
  0.6108000000040192,
  1.8508000000565517,
  4.04049999997369],
 'preorder  iterative   O(n)': [0.18049999994218524,
  0.3045999999358173,
  1.2044000000059896,
  3.1089000001429667],
 'inorder   recursive   O(n)': [0.18180000006395858,
  0.5889999999908468,
  1.1340999999447376,
  2.761899999995876],
 'inorder   iterative   O(n)': [0.3108000000793254,
  0.6372000000283151,
  0.9767999999894528,
  5.23420000013175],
 'inorder   morris      O(n)': [0.6172999999307649,
  0.802100000100836,
  1.2641999999232212,
  4.2645000000902655],
 'postorder one-stack   O(n)': [0.26370000000497384,
  0.3999999998995918,
  1.0507999998026207,
  1.916899999969246],
 'universal template    O(n)': [1.1300999999548367,
  1.3197000000673142,
  4.9539999999979045,
  5.999499999916225],
 'bfs       levels      O(n)': [0.15749999988656782,
  0.419899999997142,
  1.3073000000076718,
  1.5806999999767868]}

## 🧩 Patterns Learned

**The one-line summary:** *DFS = stack, BFS = queue; preorder/inorder/postorder differ only in **when** the node
is recorded relative to its children.*

- **Recursive DFS template.** `if not node: return` then the three lines `N`, `L`, `R` in whatever order the
  traversal names. Moving one line changes the traversal. Everything else is bookkeeping wrapped around this.
- **Preorder = stack, push right before left.** Record on pop. A stack reverses your pushes, so the child you
  want first goes on last.
- **Inorder = "dive left, pop, turn right"** (`while node or stack`). This exact loop is the backbone of the
  iterative BST problems - add a counter for k-th smallest, add a `prev` variable for validate-BST.
- **Postorder = reverse the mirrored preorder** (`N R L` reversed) when you only need the *list*; use the
  `last_visited` one-stack loop when you must *act* on nodes in true postorder.
- **The visited-flag template** (`(node, False)` / `(node, True)`) gives all three orders from one loop - the
  safest thing to reach for if you blank in an interview.
- **BFS = queue + the level-size snapshot.** `for _ in range(len(q))` taken *before* the loop is the single most
  reused line in tree problems; it is what keeps levels apart.
- **Choosing the traversal:**
  - answer built **from children** (height, sums, balanced, diameter, max path sum) -> **postorder**
  - answer about **sorted order / BST validity / k-th** -> **inorder**
  - **copy, serialize, or emit parent before children** -> **preorder**
  - **depth, levels, right-side view, shortest hops, minimum depth** -> **BFS**
- **Space trade-off:** DFS is `O(h)`, BFS is `O(w)`. Deep narrow tree -> prefer BFS. Wide shallow tree -> prefer
  DFS. Need `O(1)` and inorder specifically -> Morris.
- **Common pitfalls:**
  1. forgetting the `if not node: return` base case;
  2. pushing left before right in iterative preorder (gives a mirrored result);
  3. using `list.pop(0)` instead of `deque.popleft()` in BFS - silently `O(n²)`;
  4. reading `len(q)` *inside* the level loop, so level boundaries dissolve;
  5. iterative postorder without `last_visited` -> infinite descent down the right spine;
  6. assuming recursion is safe on a skewed tree - it is `O(n)` deep and will hit the recursion limit.
- **Related problems (LeetCode):**
  - [94. Binary Tree Inorder Traversal](https://leetcode.com/problems/binary-tree-inorder-traversal/)
  - [144. Binary Tree Preorder Traversal](https://leetcode.com/problems/binary-tree-preorder-traversal/)
  - [145. Binary Tree Postorder Traversal](https://leetcode.com/problems/binary-tree-postorder-traversal/)
  - [102. Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/)
  - [103. Binary Tree Zigzag Level Order Traversal](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/)
  - [107. Binary Tree Level Order Traversal II](https://leetcode.com/problems/binary-tree-level-order-traversal-ii/)
  - [199. Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/)
  - [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/)
  - [98. Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/)
  - [230. Kth Smallest Element in a BST](https://leetcode.com/problems/kth-smallest-element-in-a-bst/)
  - [297. Serialize and Deserialize Binary Tree](https://leetcode.com/problems/serialize-and-deserialize-binary-tree/)
  - [124. Binary Tree Maximum Path Sum](https://leetcode.com/problems/binary-tree-maximum-path-sum/)